In [1]:
import pathlib
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from notebook_init_utils import init_notebook

warnings.filterwarnings("ignore")
root_dir, in_notebook = init_notebook()

if in_notebook:
    from tqdm.notebook import tqdm
else:
    from tqdm import tqdm

results_path = pathlib.Path(root_dir, "4.linear_modeling/results/variate_importance")
results_path.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

In [2]:
df = pd.read_parquet(
    pathlib.Path(
        root_dir, "4.linear_modeling/results/linear_modeling/sc_norm.parquet"
    ).resolve(strict=True)
)
df["model"] = df["patient"] + "_" + df["treatment"] + "_" + df["feature"]
# df = df.loc[df['patient'] == "NF0014_T1"]
# df = df.loc[df['treatment'] == "Trametinib_1uM"]
df["hit"] = (df["pvalue_fdr"] < 0.05) & (df["coefficient"] > 0.1)
df

,term,patient,treatment,drug,therapeutic_category,feature,rsquared,rsquared_adj,fvalue,residual_ss,residual_variance,residual_pct,sst,sse,explained_pct,term_sum_sq,term_pct_of_total_var,pvalue,coefficient,intercept,feature_original,Compartment,Channel,Feature_type,Measurement,pvalue_fdr,model,hit
0,treatment,NF0014_T1,Trametinib_1uM,Trametinib,Kinase Inhibitor,Cell_AGP_Granularity_1,0.027613,0.021054,4.209944,2300.324722,3.879131,97.238655,2365.648428,65.323705,2.761345,2.875712,0.121561,3.895835e-01,0.343879,0.229465,Cell_AGP_Granularity_1,Cell,AGP,Granularity,1,5.066394e-01,NF0014_T1_Trametinib_1uM_Cell_AGP_Granularity_1,False
1,cell_count,NF0014_T1,Trametinib_1uM,Trametinib,Kinase Inhibitor,Cell_AGP_Granularity_1,0.027613,0.021054,4.209944,2300.324722,3.879131,97.238655,2365.648428,65.323705,2.761345,10.849663,0.458634,9.497239e-02,0.004429,0.229465,Cell_AGP_Granularity_1,Cell,AGP,Granularity,1,1.587008e-01,NF0014_T1_Trametinib_1uM_Cell_AGP_Granularity_1,False
2,organoid_count,NF0014_T1,Trametinib_1uM,Trametinib,Kinase Inhibitor,Cell_AGP_Granularity_1,0.027613,0.021054,4.209944,2300.324722,3.879131,97.238655,2365.648428,65.323705,2.761345,4.179821,0.176688,2.996773e-01,0.031940,0.229465,Cell_AGP_Granularity_1,Cell,AGP,Granularity,1,4.273731e-01,NF0014_T1_Trametinib_1uM_Cell_AGP_Granularity_1,False
3,cell_per_organoid_count,NF0014_T1,Trametinib_1uM,Trametinib,Kinase Inhibitor,Cell_AGP_Granularity_1,0.027613,0.021054,4.209944,2300.324722,3.879131,97.238655,2365.648428,65.323705,2.761345,7.689931,0.325067,1.596634e-01,-0.013376,0.229465,Cell_AGP_Granularity_1,Cell,AGP,Granularity,1,2.574373e-01,NF0014_T1_Trametinib_1uM_Cell_AGP_Granularity_1,False
4,treatment,NF0014_T1,Trametinib_1uM,Trametinib,Kinase Inhibitor,Cell_AGP_Granularity_10,0.048941,0.042526,7.628856,646.778879,1.090690,95.105907,680.061732,33.282853,4.894093,16.958087,2.493610,9.004244e-05,-0.835067,0.349158,Cell_AGP_Granularity_10,Cell,AGP,Granularity,10,3.934074e-04,NF0014_T1_Trametinib_1uM_Cell_AGP_Granularity_10,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540403,cell_per_organoid_count,SARCO361_T1,Binimetinib_1uM,Binimetinib,Kinase Inhibitor,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,0.029288,0.024935,6.728310,822.872288,0.922503,97.071188,847.699823,24.827534,2.928812,1.471706,0.173612,2.068947e-01,-0.304956,-0.226157,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei,NaN,AreaSizeShape,NoChannel,3.154770e-01,SARCO361_T1_Binimetinib_1uM_Nuclei_NoChannel_A...,False
540404,treatment,SARCO361_T1,Binimetinib_1uM,Binimetinib,Kinase Inhibitor,Nuclei_NoChannel_AreaSizeShape_Volume,0.041624,0.037327,9.685400,995.833259,1.116405,95.837556,1039.084576,43.251317,4.162444,34.264927,3.297607,3.980123e-08,-0.735306,-0.132174,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei,NaN,AreaSizeShape,NoChannel,3.417104e-07,SARCO361_T1_Binimetinib_1uM_Nuclei_NoChannel_A...,False
540405,cell_count,SARCO361_T1,Binimetinib_1uM,Binimetinib,Kinase Inhibitor,Nuclei_NoChannel_AreaSizeShape_Volume,0.041624,0.037327,9.685400,995.833259,1.116405,95.837556,1039.084576,43.251317,4.162444,0.755828,0.072740,4.108350e-01,-0.006181,-0.132174,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei,NaN,AreaSizeShape,NoChannel,5.179131e-01,SARCO361_T1_Binimetinib_1uM_Nuclei_NoChannel_A...,False
540406,organoid_count,SARCO361_T1,Binimetinib_1uM,Binimetinib,Kinase Inhibitor,Nuclei_NoChannel_AreaSizeShape_Volume,0.041624,0.037327,9.685400,995.833259,1.116405,95.837556,1039.084576,43.251317,4.162444,0.084325,0.008115,7.835088e-01,-0.004639,-0.132174,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei,NaN,AreaSizeShape,NoChannel,8.524939e-01,SARCO361_T1_Binimetinib_1uM_Nuclei_NoChannel_A...,False


In [3]:
# calculate the the number of hits for each term
term_hit_counts = df.groupby("term")["hit"].sum().reset_index()
term_hit_counts = term_hit_counts.rename(columns={"hit": "num_hits"})
term_hit_counts = term_hit_counts.sort_values(by="num_hits", ascending=False)

# calculate the total number of hits by feature (model)
feature_hit_counts = df.groupby("feature")["hit"].sum().reset_index()
feature_hit_counts = feature_hit_counts.rename(columns={"hit": "num_hits"})
feature_hit_counts = feature_hit_counts.sort_values(by="num_hits", ascending=False)
feature_hit_counts

# calculate the total number of hits for each feature and term combination
feature_term_hit_counts = df.groupby(["feature", "term"])["hit"].sum().reset_index()
feature_term_hit_counts = feature_term_hit_counts.rename(columns={"hit": "num_hits"})
feature_term_hit_counts = feature_term_hit_counts.sort_values(
    by="num_hits", ascending=False
)
feature_term_hit_counts

,feature,term,num_hits
769,Cytoplasm_AGP_Granularity_8,cell_per_organoid_count,170
57,Cell_AGP_Granularity_8,cell_per_organoid_count,168
1481,Nuclei_AGP_Granularity_8,cell_per_organoid_count,167
770,Cytoplasm_AGP_Granularity_8,organoid_count,124
58,Cell_AGP_Granularity_8,organoid_count,121
...,...,...,...
36,Cell_AGP_Granularity_3,cell_count,0
40,Cell_AGP_Granularity_4,cell_count,0
2121,Nuclei_NoChannel_AreaSizeShape_EulerNumber,cell_per_organoid_count,0
2122,Nuclei_NoChannel_AreaSizeShape_EulerNumber,organoid_count,0


### Venn and UpSet tables: which features belong to which variate group?

A feature belongs to a variate group (a model term) when it is a hit for that term (`pvalue_fdr < 0.05` and `coefficient > 0.1`, so increases only). The residual (epsilon) is not shown because every feature has one. Membership is not exclusive; overlaps are features that respond to more than one variate.

The tables are computed for two model sets:
* **original** (`sc_norm.parquet`): `treatment`, `cell_count`, `organoid_count`, `cell_per_organoid_count`.
* **technical** (`sc_norm_technical_model.parquet`): the four terms above plus the position / depth covariates.

The Venn tables use the four core terms for both model sets (a Venn is only legible up to four sets); the UpSet tables use every term of the model set. Note that the covariate coefficients are per raw unit, so the fixed `coefficient > 0.1` cut-off is not scale-free for them.

Each model set is tabulated at six scopes, and membership is recomputed within each:
* `all_models`: every model pooled.
* `per_patient_treatment`: one Venn + UpSet per patient x treatment.
* `per_patient`: one per patient, all treatments pooled.
* `per_treatment`: one per treatment, all patients pooled.
* `per_treatment_tumor_type`: one per treatment within each tumor type (`cNF` / `pNF` / `MPNST` / `Other`, looked up by patient id -- the same classification as `utils/r_plot_themes.r`'s `tumor_type_lookup`).
* `per_tumor_type`: one per tumor type (`cNF` / `pNF` / `MPNST` / `Other`), all patients and treatments pooled.

The tables are saved to `variate_hit_venn_regions_all_scopes.parquet`, `variate_hit_upset_counts_all_scopes.parquet` and `variate_hit_set_sizes_all_scopes.parquet`, and plotted in `6.plot_variate_importance` (R / ggplot2).

In [ ]:
technical_path = pathlib.Path(
    root_dir,
    "4.linear_modeling/results/linear_modeling/sc_norm_technical_model.parquet",
)

# a feature belongs to a variate group (model term) when it is a hit for that term
CORE_TERMS = ["treatment", "cell_count", "organoid_count", "cell_per_organoid_count"]
TECHNICAL_TERMS = CORE_TERMS + [
    "manhattan_distance_from_center",
    "cell_x_position",
    "cell_y_position",
    "cell_z_position",
    "cell_z_depth",
]

# the same tumor-type lookup used in 2.linear_modeling / 3.linear_modeling_technical_vars /
# 9.explore_linear_model_haystacks (and utils/r_plot_themes.r's tumor_type_lookup on the R side).
# NF0030_T1 (myopericytoma) and NF0040_T1 (schwannoma) are not NF1 nerve-sheath tumors and are
# grouped as "Other".
TUMOR_TYPE_DICT = {
    "NF0014_T1": "cNF",
    "NF0014_T2": "pNF",
    "NF0016_T1": "pNF",
    "NF0018_T6": "cNF",
    "NF0021_T1": "cNF",
    "NF0030_T1": "Other",
    "NF0035_T1": "cNF",
    "NF0037_T1": "cNF",
    "NF0040_T1": "Other",
    "NF0055_T1": "pNF",
    "SARCO219_T2": "MPNST",
    "SARCO361_T1": "MPNST",
}


def load_technical_hits(path):
    """Technical-model results with the same column names / term labels as the original model."""
    out = pd.read_parquet(path.resolve(strict=True)).rename(
        columns={
            "Metadata_Biology_PatientTumor": "patient",
            "Metadata_Experiment_Treatment": "treatment",
        }
    )
    out["term"] = out["term"].replace({"Metadata_Experiment_Treatment": "treatment"})
    out["hit"] = (out["pvalue_fdr"] < 0.05) & (out["coefficient"] > 0.1)
    return out


def slim_hits(hits):
    """Only the columns the plots need, plus tumor_type (cNF / pNF / MPNST / Other) looked up by patient id."""
    out = hits[["term", "patient", "treatment", "feature", "hit"]].copy()
    out["tumor_type"] = out["patient"].map(TUMOR_TYPE_DICT)
    return out


def build_membership(hits, terms):
    """Boolean feature x term table: True when the feature is a hit for that term in any model."""
    return (
        hits.loc[hits["term"].isin(terms)]
        .assign(hit=lambda d: d["hit"].astype(bool))
        .pivot_table(index="feature", columns="term", values="hit", aggfunc="max")
        .reindex(columns=terms)
        .fillna(False)
        .astype(bool)
    )


def venn_regions(membership, terms):
    """Feature count per Venn region; `region_key` has one bit per term (in term order)."""
    key = membership[terms].apply(
        lambda r: "".join("1" if v else "0" for v in r), axis=1
    )
    out = key.value_counts().rename_axis("region_key").reset_index(name="n_features")
    out = out.loc[out["region_key"] != "0" * len(terms)].reset_index(drop=True)
    out["variates"] = out["region_key"].map(
        lambda k: " + ".join(t for t, b in zip(terms, k) if b == "1")
    )
    return out


def upset_combinations(membership, terms):
    """Features in exactly each term combination (largest first), with one boolean `in_<term>` flag per term."""
    combos = (
        membership.groupby(terms)
        .size()
        .rename("n_features")
        .reset_index()
        .loc[lambda d: d[terms].any(axis=1)]
        .sort_values("n_features", ascending=False)
        .reset_index(drop=True)
    )
    if combos.empty:
        return combos.assign(combination=[], treatment_specific=[])
    combos["combination"] = combos[terms].apply(
        lambda r: " + ".join(t for t in terms if r[t]), axis=1
    )
    combos["treatment_specific"] = combos["treatment"] & ~combos[terms[1:]].any(axis=1)
    # term flags are boolean; prefix them so the `treatment` term does not collide
    # with the `treatment` scope identifier column
    return combos.rename(columns={t: f"in_{t}" for t in terms})


def set_sizes(membership, terms):
    """Features that are a hit for each term, plus the features that are a hit for none of the terms."""
    return pd.DataFrame(
        {
            "term": terms,
            "set_size": membership[terms].sum().to_numpy(),
            "n_total": len(membership),
            "n_none": int((~membership[terms].any(axis=1)).sum()),
        }
    )

In [ ]:
# every scope: the columns that define one Venn + UpSet pair. tumor_type = TUMOR_TYPE_DICT lookup by patient id
SCOPES = {
    "all_models": [],
    "per_patient_treatment": ["patient", "treatment"],
    "per_patient": ["patient"],
    "per_treatment": ["treatment"],
    "per_treatment_tumor_type": ["treatment", "tumor_type"],
    "per_tumor_type": ["tumor_type"],
}
venn_parquet = results_path / "variate_hit_venn_regions_all_scopes.parquet"
upset_parquet = results_path / "variate_hit_upset_counts_all_scopes.parquet"
set_size_parquet = results_path / "variate_hit_set_sizes_all_scopes.parquet"


def build_model_sets():
    """venn terms, upset terms, hits per model set (loaded only when the tables must be computed)."""
    return {
        "original": (CORE_TERMS, CORE_TERMS, slim_hits(df)),
        "technical": (
            CORE_TERMS,
            TECHNICAL_TERMS,
            slim_hits(load_technical_hits(technical_path)),
        ),
    }


def scope_frames(hits, keys):
    """Yield (key values, sub-frame) for one scope; a single all-data frame when there are no keys."""
    if not keys:
        yield (), hits
    else:
        yield from hits.groupby(keys, sort=True, observed=True)


# the tables are computed once; delete a parquet to recompute them
if all(p.exists() for p in (venn_parquet, upset_parquet, set_size_parquet)):
    print("variate hit tables already present; nothing to recompute")
else:
    venn_tables, upset_tables, size_tables = [], [], []
    for model, (venn_terms, upset_terms, hits) in build_model_sets().items():
        for scope, keys in SCOPES.items():
            groups = list(scope_frames(hits, keys))
            for values, sub in tqdm(groups, desc=f"{model} {scope}", leave=False):
                values = (values,) if isinstance(values, str) else tuple(values)
                ids = {"model_set": model, "scope": scope, **dict(zip(keys, values))}
                venn_membership = build_membership(sub, venn_terms)
                upset_membership = build_membership(sub, upset_terms)
                venn_tables.append(
                    venn_regions(venn_membership, venn_terms).assign(**ids)
                )
                size_tables.append(
                    set_sizes(venn_membership, venn_terms).assign(plot="venn", **ids)
                )
                size_tables.append(
                    set_sizes(upset_membership, upset_terms).assign(plot="upset", **ids)
                )
                combos = upset_combinations(upset_membership, upset_terms)
                if not combos.empty:
                    upset_tables.append(combos.assign(**ids))

    pd.concat(venn_tables, ignore_index=True).to_parquet(venn_parquet, index=False)
    pd.concat(upset_tables, ignore_index=True).to_parquet(upset_parquet, index=False)
    pd.concat(size_tables, ignore_index=True).to_parquet(set_size_parquet, index=False)

venn_all = pd.read_parquet(venn_parquet)
upset_all = pd.read_parquet(upset_parquet)
print(f"{len(venn_all):,} Venn region rows, {len(upset_all):,} UpSet combination rows")
upset_all.loc[
    (upset_all["model_set"] == "technical") & (upset_all["scope"] == "all_models"),
    ["combination", "n_features"],
].head(25)